In [0]:
# control engine creates a recommendation
# recommendation becomes the active command
# telemetry simulator reads that command
# next simulated telemetry changes based on that command

# will create these pieces:

# control command table - stores the command to be “applied”
# active command view - shows the latest command for the HVAC system
# enhanced telemetry simulator - reads the active command and changes simulated values
# command feedback log - records what command was applied and when

# Databricks table acts like the control system

In [0]:
%sql
-- Create control commands table
CREATE OR REPLACE TABLE hvacapp_dev3.serving.control_commands (
  command_id STRING,
  command_ts TIMESTAMP,
  recommendation_ts TIMESTAMP,
  site_id STRING,
  building_id STRING,
  hvac_system_id STRING,
  control_point_name STRING,
  current_setpoint_c DOUBLE, -- old setpoint
  commanded_setpoint_c DOUBLE, -- new setpoint
  command_status STRING, -- PENDING, APPLIED, FAILED (will simulate all commands as APPLIED)
  command_source STRING, -- RULE_BASD_CONTROLLER
  created_ts TIMESTAMP
)
USING DELTA;

In [0]:
%sql
DELETE FROM hvacapp_dev3.serving.control_commands;

INSERT INTO hvacapp_dev3.serving.control_commands
SELECT
  uuid() AS command_id,
  current_timestamp() AS command_ts,
  recommendation_ts,
  site_id,
  building_id,
  hvac_system_id,
  'chw_supply_temp_setpoint_c' AS control_point_name,
  current_setpoint_c,
  recommended_setpoint_c AS commanded_setpoint_c,
  'APPLIED' AS command_status,
  'RULE_BASED_CONTROLLER' AS command_source,
  current_timestamp() AS created_ts
FROM hvacapp_dev3.serving.control_recommendations;

In [0]:
%sql
SELECT *
FROM hvacapp_dev3.serving.control_commands
ORDER BY command_ts DESC;

In [0]:
%sql
-- # what is the latest command for this HVAC system?

CREATE OR REPLACE VIEW hvacapp_dev3.serving.active_control_command AS
SELECT
  command_id,
  command_ts,
  recommendation_ts,
  site_id,
  building_id,
  hvac_system_id,
  control_point_name,
  current_setpoint_c,
  commanded_setpoint_c,
  command_status,
  command_source,
  created_ts
FROM (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY hvac_system_id
           ORDER BY command_ts DESC, recommendation_ts DESC
         ) AS rn
  FROM hvacapp_dev3.serving.control_commands
  WHERE command_status = 'APPLIED'
) t
WHERE rn = 1;

In [0]:
%sql
-- Check active command view
SELECT *
FROM hvacapp_dev3.serving.active_control_command;

In [0]:
%sql
CREATE OR REPLACE TABLE hvacapp_dev3.serving.command_feedback_log (
  feedback_id STRING,
  feedback_ts TIMESTAMP,
  command_id STRING,
  site_id STRING,
  building_id STRING,
  hvac_system_id STRING,
  control_point_name STRING,
  commanded_setpoint_c DOUBLE,
  observed_supply_temp_c DOUBLE,
  observed_return_temp_c DOUBLE,
  observed_power_kw DOUBLE,
  feedback_status STRING,
  notes STRING,
  created_ts TIMESTAMP
)
USING DELTA;

-- after a command is applied, we want to log:

-- command was 8.0°C
-- next observed supply temp became 8.2°C
-- power was 368 kW
-- feedback status = RESPONDED

-- This is how the loop is proven working.